# 04 — PII redaction (GDPR test)

Cíl: ověřit, že citlivá data (email, telefon, IBAN, karta, rodné číslo) **nedorazí do LangSmith** v plain textu.

Postaveno na oficiální API ([docs](https://docs.langchain.com/langsmith/mask-inputs-outputs)):
`Client(anonymizer=create_anonymizer(rules))`. Anonymizer dostává payload **až po JSON serializaci**, takže projde i obsahem `HumanMessage` Pydantic objektů.

Tři vrstvy (od silné po slabou):

1. **Pure regex scrubber** (`scrub_text`) — unit-level test, žádný network call.
2. **Anonymizer** (`anonymizer`) — oficiální langsmith primitiva, naše regex pravidla.
3. **End-to-end přes agenta** — pošleme PII v dotazu, najdeme trace v LangSmith UI, ověříme že vstupy jsou maskované.

K dispozici jsou ještě dvě environmentální cesty:
- `LANGSMITH_HIDE_INPUTS=true` (full hide, traces budou prázdné) — paranoidní mód.
- Server-side rules v LangSmith UI (Project Settings → Rules) — masking po přijetí.

In [1]:
from kosik_workshop.config import load_env
load_env()

## 1. Regex scrubber — unit test

Žádný network call. Ověříme, že každý vzor PII je nahrazen placeholderem.

In [2]:
from kosik_workshop.tracing import scrub_text

cases = [
    ('email', 'Můj email je info@malikpetr.cz, napiš mi.', '[EMAIL]'),
    ('email_subdomain', 'Pošli na petr.malik+test@dev.example.co.uk dnes.', '[EMAIL]'),
    ('phone_intl', 'Volej +420 777 123 456 prosím.', '[PHONE]'),
    ('phone_local', 'Číslo 777123456 je moje.', '[PHONE]'),
    ('iban', 'Plaťte na CZ65 0800 0000 1920 0014 5399 prosím.', '[IBAN]'),
    ('card', 'Karta 4111 1111 1111 1111 propadla.', '[CARD]'),
    ('rc', 'Rodné číslo 901231/1234 mám v profilu.', '[RC]'),
    ('mixed', 'Email info@x.cz, tel +420777123456, karta 4111-1111-1111-1111.', None),
    ('clean', 'Hledám veganské mléko do 50 Kč.', None),
]

for name, text, expected in cases:
    scrubbed = scrub_text(text)
    ok = expected in scrubbed if expected else scrubbed == text or '[' in scrubbed
    flag = '✓' if ok else '✗'
    print(f'{flag} {name:20s}  {scrubbed}')

✓ email                 Můj email je [EMAIL], napiš mi.
✓ email_subdomain       Pošli na [EMAIL] dnes.
✓ phone_intl            Volej [PHONE] prosím.
✓ phone_local           Číslo [PHONE] je moje.
✓ iban                  Plaťte na [IBAN] prosím.
✓ card                  Karta [CARD]propadla.
✓ rc                    Rodné číslo [RC] mám v profilu.
✓ mixed                 Email [EMAIL], tel [PHONE], karta [CARD].
✓ clean                 Hledám veganské mléko do 50 Kč.


### Edge cases — co regex NEZACHYTÍ

Důležité vědět, kde scrubber selže — pak víš, co doplnit (presidio, ML-based detector).

In [3]:
leaks = [
    'Jméno Petr Malík v adrese.',                    # plain name — neproletí
    'Adresa: Václavské nám. 12, Praha 1.',           # adresa — neproletí
    'IČO: 12345678 a DIČ CZ12345678.',               # IČ/DIČ — neproletí
    'Heslo: SuperTajne123!',                         # heslo — neproletí
    'token: sk-proj-abc123xyz...',                   # API key — neproletí
]
for s in leaks:
    print('LEAK:', scrub_text(s))

LEAK: Jméno Petr Malík v adrese.
LEAK: Adresa: Václavské nám. 12, Praha 1.
LEAK: IČO: 12345678 a DIČ CZ12345678.
LEAK: Heslo: SuperTajne123!
LEAK: token: sk-proj-abc123xyz...


## 2. Anonymizer — oficiální LangSmith API

`langsmith.anonymizer.create_anonymizer(rules)` vrací callable, který se předává do `Client(anonymizer=...)`. LangSmith volá anonymizer **až po serializaci payloadu na JSON**, takže Pydantic objekty (`HumanMessage`) jsou v té fázi už dicty se string `content` a regexy se aplikují korektně.

To je rozdíl proti `hide_inputs`/`hide_outputs` callbackům, kde dostaneš dict s neserializovanými Pydantic objekty uvnitř — tam by ručně psaný walker tichounce přeskočil PII v message content.

In [4]:
from kosik_workshop.tracing import anonymizer
import json

# Anonymizer aplikujeme na již-serializované JSON-friendly inputs
# (jak to dělá Client interně přes _dumps_json).
serialized_inputs = {
    'question': 'Můj email je info@malikpetr.cz a karta 4111 1111 1111 1111.',
    'metadata': {
        'user_id': 'user-123',  # NE PII — nemělo by být dotčeno
        'note': 'Volá z čísla +420 777 123 456.',
    },
    'history': [
        'Předchozí dotaz: pošli na info@x.cz',
        'IBAN CZ65 0800 0000 1920 0014 5399',
    ],
}
print(json.dumps(anonymizer(serialized_inputs), indent=2, ensure_ascii=False))

{
  "question": "Můj email je [EMAIL] a karta [CARD].",
  "metadata": {
    "user_id": "user-123",
    "note": "Volá z čísla [PHONE]."
  },
  "history": [
    "Předchozí dotaz: pošli na [EMAIL]",
    "IBAN [IBAN]"
  ]
}


Stejná funkce pro outputs (LLM odpověď):

In [5]:
serialized_outputs = {
    'answer': 'Potvrzuji platbu na účet CZ65 0800 0000 1920 0014 5399. Kontakt: info@kosik.cz.',
}
print(json.dumps(anonymizer(serialized_outputs), indent=2, ensure_ascii=False))

{
  "answer": "Potvrzuji platbu na účet [IBAN]. Kontakt: [EMAIL]."
}


## 3. End-to-end přes agenta

`install_redaction()` napojí náš anonymizer (postavený přes oficiální `langsmith.anonymizer.create_anonymizer`) na každou Client instanci.

**Proč anonymizer a ne `hide_inputs` callback?**
LangSmith Client volá `hide_inputs(inputs)` na **raw dict, který může obsahovat Pydantic objekty** (`HumanMessage` apod.) — dict-walker je přeskočí a PII projde. Anonymizer naproti tomu LangSmith **nejprve serializuje na JSON** (`_dumps_json`), takže k regex pravidlům dorazí čistý dict s `messages[].content` jako string.

**Pořadí MUSÍ být:**
1. `install_redaction()` (PŘED jakýmkoli LangChain voláním)
2. `build_agent()`
3. `agent.invoke(...)`

`install_redaction()` navíc resetuje `langsmith.run_trees._CLIENT` cache, aby případný pre-existing klient bez anonymizeru nezůstal v paměti.

In [6]:
from kosik_workshop.tracing import install_redaction

install_redaction()
print('Redaction installed.')

# Sanity — cached client by měl mít _anonymizer (nikoli _hide_inputs callable).
import langsmith.run_trees as rt
client = rt.get_cached_client()
print('anonymizer attached:', client._anonymizer is not None)

Redaction installed.
anonymizer attached: True


In [7]:
import uuid
from langchain_core.messages import HumanMessage
from kosik_workshop.agent import build_agent

agent = build_agent()
thread_id = str(uuid.uuid4())
print('thread_id pro LangSmith filter:', thread_id)

thread_id pro LangSmith filter: ae512ddd-c7ae-463f-a749-38f9e313d329


In [8]:
pii_question = (
    'Hledám veganské mléko do 50 Kč. Pošlete potvrzení na info@malikpetr.cz '
    'nebo zavolejte na +420 777 123 456.'
)
config = {
    'configurable': {'thread_id': thread_id},
    'metadata': {
        'thread_id': thread_id,
        'session_id': thread_id,
        'user_id': 'redaction-test-user',
    },
    'tags': ['surface:notebook', 'feature:redaction-test'],
}
result = agent.invoke({'messages': [HumanMessage(content=pii_question)]}, config=config)
print('Odpověď (lokálně nemaskovaná, agent musí vědět co odpovídá):')
print(result['messages'][-1].content[:500])
print(f'\nV LangSmith najdi přes session_id={thread_id}')

Odpověď (lokálně nemaskovaná, agent musí vědět co odpovídá):
Bohužel jsem nenašel žádné veganské mléko do 50 Kč. Mohu vám nabídnout alternativy, jako jsou veganské produkty v jiných kategoriích. Například:

- **Mlýn Pernštejn Celozrnná mouka 1 kg** (`mlyn-pernstejn-celozrnna-mouka-1-kg`)
  - Cena: 30 Kč / kg
  - Důvod: Celozrnná mouka je veganská a cenově dostupná.

- **Odkolek Chléb žitný 400 g** (`odkolek-chleb-zitny-400-g`)
  - Cena: 45 Kč / g
  - Důvod: Žitný chléb je veganský a skvělý pro snídani.

Pokud máte zájem o jinou kategorii nebo jiný typ pro

V LangSmith najdi přes session_id=ae512ddd-c7ae-463f-a749-38f9e313d329


### Verifikace v LangSmith UI

1. Otevři https://eu.smith.langchain.com → projekt `kosik-workshop`.
2. Filter → Metadata → `thread_id` = (zkopíruj UUID vytištěné výše).
3. Klikni na trace → tab **Inputs**. Měl bys vidět něco jako:
   ```
   {
     "messages": [
       {"content": "...info@malikpetr.cz...", ...}   ← PŘED redaction (špatně)
       {"content": "...[EMAIL]...", ...}              ← PO redaction (správně)
     ]
   }
   ```
4. Pokud vidíš plain `info@malikpetr.cz` → redaction není napojený. Restartni kernel a spusť cell s `langsmith.client._CLIENT = redacting` PŘED `agent.invoke(...)`.

**Důležité:** redacting client mění JEN to, co jde do LangSmith. Lokálně v notebooku má `result['messages'][-1].content` původní (nemaskovanou) odpověď — to je správně, agent musí vědět, na co odpovídá.

## 4. Negative test — bez redaction (kontrolní)

Pro porovnání chceme druhý trace **bez** redaction. Protože monkey-patch už proběhl globálně, jediná spolehlivá cesta je vytvořit explicit Client instance s `hide_inputs=None` (parametr má default `None`, ale kvůli patchi by se nastavil — proto explicit `False` ho zakáže) a předat ho LangChain tracer.

**Jednodušší alternativa:** restart kernelu, přeskočit cell s `install_redaction()`, spustit jen 4. sekci. Tím získáš čistý baseline trace.

Cell níže je hack: vytvoříme tracer s vlastním nepatchovaným klientem.

In [9]:
from langsmith import Client
from langchain_core.tracers.langchain import LangChainTracer

# Bypass — patched __init__ defaultne anonymizer, ale explicit None ho přebije.
unhooked_client = Client(anonymizer=None)
unhooked_client._anonymizer = None  # belt & suspenders
control_tracer = LangChainTracer(client=unhooked_client)

control_thread_id = str(uuid.uuid4())
print('control thread_id (BEZ redaction):', control_thread_id)

control_config = {
    'configurable': {'thread_id': control_thread_id},
    'metadata': {
        'thread_id': control_thread_id,
        'session_id': control_thread_id,
        'user_id': 'redaction-control',
    },
    'tags': ['surface:notebook', 'feature:redaction-control'],
    'callbacks': [control_tracer],
}
_ = agent.invoke({'messages': [HumanMessage(content=pii_question)]}, config=control_config)
print('Hotovo. V LangSmith porovnej oba thread_id.')

control thread_id (BEZ redaction): a940e517-10c3-4b05-b961-88a411fa7bdc
Hotovo. V LangSmith porovnej oba thread_id.


## 5. Souhrn co jsi otestoval

| Vrstva | Co maskuje | Kde běží | Jak ověřit |
|---|---|---|---|
| `scrub_text` | regex placeholders | inline v procesu | unit test (sekce 1) |
| `hide_inputs/outputs` | rekurzivně dict/list/str | hook v Client | dict diff (sekce 2) |
| Redacting client | vše co jde do LangSmith | LangChain tracer | LangSmith UI diff (sekce 3 vs 4) |
| `LANGSMITH_HIDE_INPUTS=true` | celý input → `[REDACTED]` | env var, paranoidní | trace má prázdné inputs |
| Server-side rules | regex po přijetí | LangSmith server | Project Settings UI |

## Co by se mělo přidat před produkcí

1. **Presidio** pro ML-based PII (jména, adresy, IČ/DIČ — regex je nezachytí).
2. **Pytest gate** — test, který do agenta pošle známé PII a `mock`em zachytí, co by šlo do LangSmith klienta. Selže CI, kdyby někdo redaction omylem vypnul.
3. **Per-user delete script** — najdi traces přes `metadata.user_id` filter, smaž je. Pro DELETE GDPR requesty.
4. **Region pinning** — `eu.smith.langchain.com` máš, ale ověř v Project Settings, že region je opravdu EU.